# Replay verification, no model

Applies 8 patches (that resolved LOCALLY) to the base snapshots, applies the official
test_patch, and runs exactly the test nodes swegemma would run. CPU only.

Purpose: is the local sandbox the reason these tasks resolve locally but not on Kaggle?


In [ ]:

import base64, json, os, subprocess, sys
from pathlib import Path

print('input:', sorted(os.listdir('/kaggle/input')))
TASKS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'tasks.jsonl' in files:
        TASKS = Path(root) / 'tasks.jsonl'
        break
SNAPS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith('.tgz') for f in files):
        SNAPS = Path(root)
        break
print('tasks:', TASKS)
print('snaps:', SNAPS)
assert TASKS and SNAPS

PATCHES = json.loads("""{"fastapi_9753": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weSBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQppbmRleCAxMzQ4NmRkMTguLjE4ZmM0Y2MyNCAxMDA2NDQKLS0tIGEvZmFzdGFwaS9kZXBlbmRlbmNpZXMvbW9kZWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL21vZGVscy5weQpAQCAtMSw3ICsxLDcgQEAKIGltcG9ydCBpbnNwZWN0CiBpbXBvcnQgc3lzCiBmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCi1mcm9tIGZ1bmN0b29scyBpbXBvcnQgY2FjaGVkX3Byb3BlcnR5Citmcm9tIGZ1bmN0b29scyBpbXBvcnQgY2FjaGVkX3Byb3BlcnR5LCBwYXJ0aWFsCiBmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZSwgTGlzdCwgT3B0aW9uYWwsIFNlcXVlbmNlLCBVbmlvbgogCiBmcm9tIGZhc3RhcGkuX2NvbXBhdCBpbXBvcnQgTW9kZWxGaWVsZApAQCAtNzksNyArNzksMTQgQEAgY2xhc3MgRGVwZW5kYW50OgogICAgIGRlZiBfdW53cmFwcGVkX2NhbGwoc2VsZikgLT4gQW55OgogICAgICAgICBpZiBzZWxmLmNhbGwgaXMgTm9uZToKICAgICAgICAgICAgIHJldHVybiBzZWxmLmNhbGwgICMgcHJhZ21hOiBubyBjb3ZlcgotICAgICAgICByZXR1cm4gaW5zcGVjdC51bndyYXAoc2VsZi5jYWxsKQorICAgICAgICAKKyAgICAgICAgIyBSZWN1cnNpdmVseSB1bndyYXAgcGFydGlhbCBvYmplY3RzIGFuZCBkZWNvcmF0b3JzCisgICAgICAgIHdoaWxlIGhhc2F0dHIoc2VsZi5jYWxsLCAnX193cmFwcGVkX18nKToKKyAgICAgICAgICAgIHNlbGYuY2FsbCA9IHNlbGYuY2FsbC5fX3dyYXBwZWRfXworICAgICAgICAKKyAgICAgICAgaWYgaXNpbnN0YW5jZShzZWxmLmNhbGwsIHBhcnRpYWwpOgorICAgICAgICAgICAgcmV0dXJuIHNlbGYuY2FsbC5mdW5jCisgICAgICAgIHJldHVybiBzZWxmLmNhbGwKIAogICAgIEBjYWNoZWRfcHJvcGVydHkKICAgICBkZWYgaXNfZ2VuX2NhbGxhYmxlKHNlbGYpIC0+IGJvb2w6Cg==", "requests_7309": "ZGlmZiAtLWdpdCBhL3NyYy9yZXF1ZXN0cy91dGlscy5weSBiL3NyYy9yZXF1ZXN0cy91dGlscy5weQppbmRleCA1NDk1OWJiOC4uODA1Y2Y5ZDIgMTAwNjQ0Ci0tLSBhL3NyYy9yZXF1ZXN0cy91dGlscy5weQorKysgYi9zcmMvcmVxdWVzdHMvdXRpbHMucHkKQEAgLTUxNywxMiArNTE3LDExIEBAIGRlZiBfcGFyc2VfY29udGVudF90eXBlX2hlYWRlcihoZWFkZXIpOgogICAgIGZvciBwYXJhbSBpbiBwYXJhbXM6CiAgICAgICAgIHBhcmFtID0gcGFyYW0uc3RyaXAoKQogICAgICAgICBpZiBwYXJhbToKLSAgICAgICAgICAgIGtleSwgdmFsdWUgPSBwYXJhbSwgVHJ1ZQogICAgICAgICAgICAgaW5kZXhfb2ZfZXF1YWxzID0gcGFyYW0uZmluZCgiPSIpCiAgICAgICAgICAgICBpZiBpbmRleF9vZl9lcXVhbHMgIT0gLTE6CiAgICAgICAgICAgICAgICAga2V5ID0gcGFyYW1bOmluZGV4X29mX2VxdWFsc10uc3RyaXAoaXRlbXNfdG9fc3RyaXApCiAgICAgICAgICAgICAgICAgdmFsdWUgPSBwYXJhbVtpbmRleF9vZl9lcXVhbHMgKyAxIDpdLnN0cmlwKGl0ZW1zX3RvX3N0cmlwKQotICAgICAgICAgICAgcGFyYW1zX2RpY3Rba2V5Lmxvd2VyKCldID0gdmFsdWUKKyAgICAgICAgICAgICAgICBwYXJhbXNfZGljdFtrZXkubG93ZXIoKV0gPSB2YWx1ZQogICAgIHJldHVybiBjb250ZW50X3R5cGUsIHBhcmFtc19kaWN0CiAKIAo=", "requests_7315": "ZGlmZiAtLWdpdCBhL3NyYy9yZXF1ZXN0cy9hZGFwdGVycy5weSBiL3NyYy9yZXF1ZXN0cy9hZGFwdGVycy5weQppbmRleCA5OGY3NDQ2NS4uMjc5MTk0NjggMTAwNjQ0Ci0tLSBhL3NyYy9yZXF1ZXN0cy9hZGFwdGVycy5weQorKysgYi9zcmMvcmVxdWVzdHMvYWRhcHRlcnMucHkKQEAgLTU0NSw4ICs1NDUsNiBAQCBjbGFzcyBIVFRQQWRhcHRlcihCYXNlQWRhcHRlcik6CiAgICAgICAgICAgICB1c2luZ19zb2Nrc19wcm94eSA9IHByb3h5X3NjaGVtZS5zdGFydHN3aXRoKCJzb2NrcyIpCiAKICAgICAgICAgdXJsID0gcmVxdWVzdC5wYXRoX3VybAotICAgICAgICBpZiB1cmwuc3RhcnRzd2l0aCgiLy8iKTogICMgRG9uJ3QgY29uZnVzZSB1cmxsaWIzCi0gICAgICAgICAgICB1cmwgPSBmIi97dXJsLmxzdHJpcCgnLycpfSIKIAogICAgICAgICBpZiBpc19wcm94aWVkX2h0dHBfcmVxdWVzdCBhbmQgbm90IHVzaW5nX3NvY2tzX3Byb3h5OgogICAgICAgICAgICAgdXJsID0gdXJsZGVmcmFnYXV0aChyZXF1ZXN0LnVybCkK", "requests_7427": "ZGlmZiAtLWdpdCBhL3NyYy9yZXF1ZXN0cy91dGlscy5weSBiL3NyYy9yZXF1ZXN0cy91dGlscy5weQppbmRleCBlNjlhZDBkNy4uMjc2MGJlZmIgMTAwNjQ0Ci0tLSBhL3NyYy9yZXF1ZXN0cy91dGlscy5weQorKysgYi9zcmMvcmVxdWVzdHMvdXRpbHMucHkKQEAgLTg1MSw3ICs4NTEsNyBAQCBkZWYgc2hvdWxkX2J5cGFzc19wcm94aWVzKHVybDogc3RyLCBub19wcm94eTogc3RyIHwgTm9uZSkgLT4gYm9vbDoKICAgICAgICAgICAgICAgICBob3N0X3dpdGhfcG9ydCArPSBmIjp7cGFyc2VkLnBvcnR9IgogCiAgICAgICAgICAgICBmb3IgaG9zdCBpbiBub19wcm94eV9ob3N0czoKLSAgICAgICAgICAgICAgICBpZiBob3N0bmFtZS5lbmRzd2l0aChob3N0KSBvciBob3N0X3dpdGhfcG9ydC5lbmRzd2l0aChob3N0KToKKyAgICAgICAgICAgICAgICBpZiBob3N0bmFtZSA9PSBob3N0IG9yIGhvc3RuYW1lLnN0YXJ0c3dpdGgoaG9zdCArICcuJyk6CiAgICAgICAgICAgICAgICAgICAgICMgVGhlIFVSTCBkb2VzIG1hdGNoIHNvbWV0aGluZyBpbiBub19wcm94eSwgc28gd2UgZG9uJ3Qgd2FudAogICAgICAgICAgICAgICAgICAgICAjIHRvIGFwcGx5IHRoZSBwcm94aWVzIG9uIHRoaXMgVVJMLgogICAgICAgICAgICAgICAgICAgICByZXR1cm4gVHJ1ZQo=", "fastapi_14616": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5IGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKaW5kZXggZTVhYzUxZDUzLi4wMDg2YzNmNzYgMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC03MzEsNyArNzMxLDcgQEAgZGVmIF9nZXRfbXVsdGlkaWN0X3ZhbHVlKAogKSAtPiBBbnk6CiAgICAgYWxpYXMgPSBhbGlhcyBvciBnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZCkKICAgICBpZiBpc19zZXF1ZW5jZV9maWVsZChmaWVsZCkgYW5kIGlzaW5zdGFuY2UodmFsdWVzLCAoSW1tdXRhYmxlTXVsdGlEaWN0LCBIZWFkZXJzKSk6Ci0gICAgICAgIHZhbHVlID0gdmFsdWVzLmdldGxpc3QoYWxpYXMpCisgICAgICAgIHZhbHVlID0gdmFsdWVzLmdldChhbGlhcykKICAgICBlbHNlOgogICAgICAgICB2YWx1ZSA9IHZhbHVlcy5nZXQoYWxpYXMsIE5vbmUpCiAgICAgaWYgKApAQCAtNzk3LDcgKzc5Nyw3IEBAIGRlZiByZXF1ZXN0X3BhcmFtc190b19hcmdzKAogICAgIGZvciBrZXkgaW4gcmVjZWl2ZWRfcGFyYW1zLmtleXMoKToKICAgICAgICAgaWYga2V5IG5vdCBpbiBwcm9jZXNzZWRfa2V5czoKICAgICAgICAgICAgIGlmIGhhc2F0dHIocmVjZWl2ZWRfcGFyYW1zLCAiZ2V0bGlzdCIpOgotICAgICAgICAgICAgICAgIHZhbHVlID0gcmVjZWl2ZWRfcGFyYW1zLmdldGxpc3Qoa2V5KQorICAgICAgICAgICAgICAgIHZhbHVlID0gcmVjZWl2ZWRfcGFyYW1zLmdldChrZXkpCiAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbGlzdCkgYW5kIChsZW4odmFsdWUpID09IDEpOgogICAgICAgICAgICAgICAgICAgICBwYXJhbXNfdG9fcHJvY2Vzc1trZXldID0gdmFsdWVbMF0KICAgICAgICAgICAgICAgICBlbHNlOgpAQCAtOTE1LDcgKzkxNSw3IEBAIGFzeW5jIGRlZiBfZXh0cmFjdF9mb3JtX2JvZHkoCiAgICAgZmllbGRfYWxpYXNlcyA9IHtnZXRfdmFsaWRhdGlvbl9hbGlhcyhmaWVsZCkgZm9yIGZpZWxkIGluIGJvZHlfZmllbGRzfQogICAgIGZvciBrZXkgaW4gcmVjZWl2ZWRfYm9keS5rZXlzKCk6CiAgICAgICAgIGlmIGtleSBub3QgaW4gZmllbGRfYWxpYXNlczoKLSAgICAgICAgICAgIHBhcmFtX3ZhbHVlcyA9IHJlY2VpdmVkX2JvZHkuZ2V0bGlzdChrZXkpCisgICAgICAgICAgICBwYXJhbV92YWx1ZXMgPSByZWNlaXZlZF9ib2R5LmdldChrZXkpCiAgICAgICAgICAgICBpZiBsZW4ocGFyYW1fdmFsdWVzKSA9PSAxOgogICAgICAgICAgICAgICAgIHZhbHVlc1trZXldID0gcGFyYW1fdmFsdWVzWzBdCiAgICAgICAgICAgICBlbHNlOgo=", "fastapi_9425": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5IGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKaW5kZXggOGRkMGExNGM4Li5jMThjYWUwNzIgMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC0yNjEsNyArMjYxLDcgQEAgZGVmIGdldF90eXBlZF9yZXR1cm5fYW5ub3RhdGlvbihjYWxsOiBDYWxsYWJsZVsuLi4sIEFueV0pIC0+IEFueToKICAgICBzaWduYXR1cmUgPSBpbnNwZWN0LnNpZ25hdHVyZShjYWxsKQogICAgIGFubm90YXRpb24gPSBzaWduYXR1cmUucmV0dXJuX2Fubm90YXRpb24KIAotICAgIGlmIGFubm90YXRpb24gaXMgaW5zcGVjdC5TaWduYXR1cmUuZW1wdHk6CisgICAgaWYgYW5ub3RhdGlvbiBpcyBpbnNwZWN0LlNpZ25hdHVyZS5lbXB0eSBvciBhbm5vdGF0aW9uID09ICJOb25lIjoKICAgICAgICAgcmV0dXJuIE5vbmUKIAogICAgIGdsb2JhbG5zID0gZ2V0YXR0cihjYWxsLCAiX19nbG9iYWxzX18iLCB7fSkK", "rich_3067": "ZGlmZiAtLWdpdCBhL3JpY2gvaGlnaGxpZ2h0ZXIucHkgYi9yaWNoL2hpZ2hsaWdodGVyLnB5CmluZGV4IGMyNjQ2Nzk0Li4yNzcxNGIyNSAxMDA2NDQKLS0tIGEvcmljaC9oaWdobGlnaHRlci5weQorKysgYi9yaWNoL2hpZ2hsaWdodGVyLnB5CkBAIC05OCw3ICs5OCw3IEBAIGNsYXNzIFJlcHJIaWdobGlnaHRlcihSZWdleEhpZ2hsaWdodGVyKToKICAgICAgICAgICAgIHIiKD9QPG51bWJlcj4oPzwhXHcpXC0/WzAtOV0rXC4/WzAtOV0qKGVbLStdP1xkKz8pP1xifDB4WzAtOWEtZkEtRl0qKSIsCiAgICAgICAgICAgICByIig/UDxwYXRoPlxCKC9bLVx3Ll8rXSspKlwvKSg/UDxmaWxlbmFtZT5bLVx3Ll8rXSopPyIsCiAgICAgICAgICAgICByIig/PCFbXFxcd10pKD9QPHN0cj5iPycnJy4qPyg/PCFcXCknJyd8Yj8nLio/KD88IVxcKSd8Yj9cIlwiXCIuKj8oPzwhXFwpXCJcIlwifGI/XCIuKj8oPzwhXFwpXCIpIiwKLSAgICAgICAgICAgIHIiKD9QPHVybD4oZmlsZXxodHRwc3xodHRwfHdzfHdzcyk6Ly9bLTAtOWEtekEtWiRfKyFgKCksLj8vOzomPSUjXSopIiwKKyAgICAgICAgICAgIHIiKD9QPHVybD4oZmlsZXxodHRwc3xodHRwfHdzfHdzcyk6Ly9bLTAtOWEtekEtWiRfKyFgKCksLj8vOzomPSUjfl0qKSIsCiAgICAgICAgICksCiAgICAgXQogCg==", "rich_3882": "ZGlmZiAtLWdpdCBhL3JpY2gvcHJvbXB0LnB5IGIvcmljaC9wcm9tcHQucHkKaW5kZXggYzdjZjI1YmEuLmFlOTRkOWJiIDEwMDY0NAotLS0gYS9yaWNoL3Byb21wdC5weQorKysgYi9yaWNoL3Byb21wdC5weQpAQCAtMjYyLDcgKzI2Miw3IEBAIGNsYXNzIFByb21wdEJhc2UoR2VuZXJpY1tQcm9tcHRUeXBlXSk6CiAgICAgICAgICAgICB2YWx1ZSAoc3RyKTogU3RyaW5nIGVudGVyZWQgYnkgdXNlci4KICAgICAgICAgICAgIGVycm9yIChJbnZhbGlkUmVzcG9uc2UpOiBFeGNlcHRpb24gaW5zdGFuY2UgdGhlIGluaXRpYXRlZCB0aGUgZXJyb3IuCiAgICAgICAgICIiIgotICAgICAgICBzZWxmLmNvbnNvbGUucHJpbnQoZXJyb3IpCisgICAgICAgIHNlbGYuY29uc29sZS5wcmludChlcnJvciwgbWFya3VwPVRydWUpCiAKICAgICBkZWYgcHJlX3Byb21wdChzZWxmKSAtPiBOb25lOgogICAgICAgICAiIiJIb29rIHRvIGRpc3BsYXkgc29tZXRoaW5nIGJlZm9yZSB0aGUgcHJvbXB0LiIiIgo="}""")
PD = Path('/kaggle/working/patches'); PD.mkdir(parents=True, exist_ok=True)
for tid, b64 in PATCHES.items():
    (PD / f'{tid}.patch').write_bytes(base64.b64decode(b64))
print('patches written:', len(PATCHES))

RUNNER = Path('/kaggle/working/replay_verify2.py')
RUNNER.write_bytes(base64.b64decode("""IiIiRmFpdGhmdWwgcmVwbGF5OiBtaXJyb3JzIHN3ZWdlbW1hJ3Mgb3duIHZlcmlmaWNhdGlvbiBydWxlLgoKc3dlZ2VtbWEgZGVyaXZlcyB0aGUgcmVxdWlyZWQgdGVzdCBub2RlcyBmcm9tIHRoZSB0ZXN0X3BhdGNoCihgX2V4dHJhY3RfdGVzdF9mdW5jdGlvbnNfZnJvbV9wYXRjaGApIGFuZCBydW5zIG9ubHkgdGhvc2UsIGZhaWxpbmcgb24KZmFpbGVkL2Vycm9yL3NraXBwZWQuIFRoaXMgc2NyaXB0IGRvZXMgZXhhY3RseSB0aGUgc2FtZSwgd2l0aCBubyBtb2RlbCBpbnZvbHZlZCwKc28gdGhlIHNhbWUgcGF0Y2hlcyBjYW4gYmUgcmVwbGF5ZWQgaW4gYW55IGVudmlyb25tZW50LgoiIiIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKVEFTS1MgPSBQYXRoKHN5cy5hcmd2WzFdKQpTTkFQUyA9IFBhdGgoc3lzLmFyZ3ZbMl0pClBBVENIRElSID0gUGF0aChzeXMuYXJndlszXSkKSURTID0gc3lzLmFyZ3ZbNDpdCk9VVCA9IFBhdGgob3MuZW52aXJvbi5nZXQoJ1JFUExBWV9PVVQnLCAnL291dC9yZXBsYXlfcmVzdWx0Lmpzb24nKSkKCnRhc2tzID0ge2pzb24ubG9hZHMobClbJ2luc3RhbmNlX2lkJ106IGpzb24ubG9hZHMobCkKICAgICAgICAgZm9yIGwgaW4gVEFTS1MucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpLnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCl9ClRFU1RGSUxFID0gcmUuY29tcGlsZShyJ15cK1wrXCsgYi8oLio/KSQnLCByZS5NKQoKCmRlZiBleHRyYWN0X3Rlc3RzKHRlc3RfcGF0Y2g6IHN0cik6CiAgICAiIiJCeXRlLWlkZW50aWNhbCBsb2dpYyB0byBzd2VnZW1tYS5oYXJuZXNzLnZlcmlmaWNhdGlvbi5fZXh0cmFjdF90ZXN0X2Z1bmN0aW9uc19mcm9tX3BhdGNoLiIiIgogICAgaWYgbm90IHRlc3RfcGF0Y2g6CiAgICAgICAgcmV0dXJuIFtdCiAgICBhZGRlZCA9IHJlLmZpbmRhbGwocideXCtccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkKICAgIGlmIGFkZGVkOgogICAgICAgIHJldHVybiBsaXN0KGRpY3QuZnJvbWtleXMoYWRkZWQpKQogICAgcmVtb3ZlZCA9IHNldChyZS5maW5kYWxsKHInXi1ccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkpCiAgICBodW5rX2hlYWRlcnMgPSByZS5maW5kYWxsKHInXkBAW15AXStAQFxzKig/OmFzeW5jXHMrKT9kZWZccysodGVzdF9bQS1aYS16MC05X10rKVxiJywgdGVzdF9wYXRjaCwgcmUuTVVMVElMSU5FKQogICAgY29udGV4dF9kZWZzID0gcmUuZmluZGFsbChyJ14gXHMqKD86YXN5bmNccyspP2RlZlxzKyh0ZXN0X1tBLVphLXowLTlfXSspXHMqXCgnLCB0ZXN0X3BhdGNoLCByZS5NVUxUSUxJTkUpCiAgICBjb250ZXh0X3NldCA9IHNldChjb250ZXh0X2RlZnMpCiAgICBjb21iaW5lZCA9IGxpc3QoZGljdC5mcm9ta2V5cyhodW5rX2hlYWRlcnMgKyBjb250ZXh0X2RlZnMpKQogICAgcmV0dXJuIFtmbiBmb3IgZm4gaW4gY29tYmluZWQgaWYgZm4gbm90IGluIHJlbW92ZWQgYW5kIG5vdCAoCiAgICAgICAgZm4gbm90IGluIGNvbnRleHRfc2V0IGFuZCAoYW55KHIuc3RhcnRzd2l0aChmbikgZm9yIHIgaW4gcmVtb3ZlZCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBhbnkoYyAhPSBmbiBhbmQgYy5zdGFydHN3aXRoKGZuKSBmb3IgYyBpbiBjb250ZXh0X3NldCkpKV0KCgpkZWYgc2goY21kLCBjd2QpOgogICAgcmV0dXJuIHN1YnByb2Nlc3MucnVuKGNtZCwgY3dkPXN0cihjd2QpLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZW5jb2Rpbmc9J3V0Zi04JywgZXJyb3JzPSdyZXBsYWNlJywgdGltZW91dD0xODAwKQoKCnJlc3VsdHMgPSBbXQpmb3IgdGlkIGluIElEUzoKICAgIHJlYyA9IGRpY3QodGFzaz10aWQpCiAgICBzbmFwLCBwYXRjaCA9IFNOQVBTIC8gZid7dGlkfS50Z3onLCBQQVRDSERJUiAvIGYne3RpZH0ucGF0Y2gnCiAgICB0ID0gdGFza3MuZ2V0KHRpZCwge30pCiAgICB0cF90ZXh0ID0gdC5nZXQoJ3Rlc3RfcGF0Y2gnKSBvciAnJwogICAgaWYgbm90IHNuYXAuZXhpc3RzKCk6CiAgICAgICAgcmVjWydlcnJvciddID0gJ25vIHNuYXBzaG90JwogICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICBjb250aW51ZQogICAgd29yayA9IFBhdGgodGVtcGZpbGUubWtkdGVtcChwcmVmaXg9ZidyX3t0aWR9XycpKQogICAgdHJ5OgogICAgICAgIGlmIHNoKFsndGFyJywgJ3h6ZicsIHN0cihzbmFwKSwgJy1DJywgc3RyKHdvcmspXSwgd29yaykucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICByZWNbJ2Vycm9yJ10gPSAnZXh0cmFjdCBmYWlsZWQnCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBwYXRjaC5leGlzdHMoKToKICAgICAgICAgICAgcCA9IHNoKFsnZ2l0JywgJ2FwcGx5JywgJy0td2hpdGVzcGFjZT1ub3dhcm4nLCBzdHIocGF0Y2gucmVzb2x2ZSgpKV0sIHdvcmspCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfcmMnXSA9IHAucmV0dXJuY29kZQogICAgICAgICAgICBpZiBwLnJldHVybmNvZGUgIT0gMDoKICAgICAgICAgICAgICAgIHAgPSBzaChbJ2dpdCcsICdhcHBseScsICctLTN3YXknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cihwYXRjaC5yZXNvbHZlKCkpXSwgd29yaykKICAgICAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfM3dheV9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfdGFpbCddID0gKChwLnN0ZGVyciBvciAnJykgKyAocC5zdGRvdXQgb3IgJycpKVstMjAwOl0KICAgICAgICB0cCA9IHdvcmsucGFyZW50IC8gZid7dGlkfV90ZXN0LnBhdGNoJwogICAgICAgIHRwLndyaXRlX3RleHQodHBfdGV4dCBpZiB0cF90ZXh0LmVuZHN3aXRoKCdcbicpIGVsc2UgdHBfdGV4dCArICdcbicsIGVuY29kaW5nPSd1dGYtOCcpCiAgICAgICAgcCA9IHNoKFsnZ2l0JywgJ2FwcGx5JywgJy0td2hpdGVzcGFjZT1ub3dhcm4nLCBzdHIodHApXSwgd29yaykKICAgICAgICByZWNbJ3Rlc3RfcGF0Y2hfcmMnXSA9IHAucmV0dXJuY29kZQogICAgICAgIGlmIHAucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICBwID0gc2goWydnaXQnLCAnYXBwbHknLCAnLS0zd2F5JywgJy0td2hpdGVzcGFjZT1ub3dhcm4nLCBzdHIodHApXSwgd29yaykKICAgICAgICAgICAgcmVjWyd0ZXN0X3BhdGNoXzN3YXlfcmMnXSA9IHAucmV0dXJuY29kZQogICAgICAgICAgICByZWNbJ3Rlc3RfcGF0Y2hfdGFpbCddID0gKChwLnN0ZGVyciBvciAnJykgKyAocC5zdGRvdXQgb3IgJycpKVstMjAwOl0KCiAgICAgICAgZmlsZXMgPSBzb3J0ZWQoe2Yuc3RyaXAoKSBmb3IgZiBpbiBURVNURklMRS5maW5kYWxsKHRwX3RleHQpfSkKICAgICAgICBmbnMgPSBleHRyYWN0X3Rlc3RzKHRwX3RleHQpCiAgICAgICAgcmVjWyd0ZXN0X2ZpbGVzJ10gPSBmaWxlcwogICAgICAgIHJlY1sndGVzdF9mbnMnXSA9IGZucwogICAgICAgIG5vZGVzID0gW2Yne2Z9Ojp7Zm59JyBmb3IgZiBpbiBmaWxlcyBmb3IgZm4gaW4gZm5zXSBvciBmaWxlcwogICAgICAgIGlmIG5vdCBub2RlczoKICAgICAgICAgICAgcmVjWydlcnJvciddID0gJ25vIG5vZGVzJwogICAgICAgICAgICByZXN1bHRzLmFwcGVuZChyZWMpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcCA9IHNoKFtzeXMuZXhlY3V0YWJsZSwgJy1tJywgJ3B5dGVzdCcsICctcScsICctLW5vLWhlYWRlcicsICctcCcsICdubzpjYWNoZXByb3ZpZGVyJ10gKyBub2Rlcywgd29yaykKICAgICAgICB0YWlsID0gKHAuc3Rkb3V0IG9yICcnKSArIChwLnN0ZGVyciBvciAnJykKICAgICAgICBtID0gcmUuZmluZGFsbChyJyhcZCspXHMrKHBhc3NlZHxmYWlsZWR8ZXJyb3J8ZXJyb3JzfHNraXBwZWQpJywgdGFpbCkKICAgICAgICByZWNbJ2NvdW50cyddID0ge2s6IGludCh2KSBmb3IgdiwgayBpbiBtfQogICAgICAgIHJlY1snbm9kZXMnXSA9IG5vZGVzCiAgICAgICAgcmVjWydweXRlc3RfdGFpbCddID0gdGFpbFstMzAwOl0KICAgICAgICBwYXNzZWQgPSByZWNbJ2NvdW50cyddLmdldCgncGFzc2VkJywgMCkKICAgICAgICBiYWQgPSBzdW0ocmVjWydjb3VudHMnXS5nZXQoaywgMCkgZm9yIGsgaW4gKCdmYWlsZWQnLCAnZXJyb3InLCAnZXJyb3JzJywgJ3NraXBwZWQnKSkKICAgICAgICByZWNbJ3Jlc29sdmVkX2hlcmUnXSA9IGJvb2wocGFzc2VkID4gMCBhbmQgYmFkID09IDApCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmVjWydlcnJvciddID0gZid7dHlwZShlKS5fX25hbWVfX306IHtlfScKICAgIGZpbmFsbHk6CiAgICAgICAgc2h1dGlsLnJtdHJlZSh3b3JrLCBpZ25vcmVfZXJyb3JzPVRydWUpCiAgICByZXN1bHRzLmFwcGVuZChyZWMpCgpPVVQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKT1VULndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXN1bHRzLCBpbmRlbnQ9MSksIGVuY29kaW5nPSd1dGYtOCcpCnByaW50KCclLTIycyAlLTdzICUtN3MgJS05cyAlcycgJSAoJ3Rhc2snLCAnYV9yYycsICd0X3JjJywgJ3Jlc29sdmVkJywgJ2NvdW50cyAvIGVycicpKQpmb3IgciBpbiByZXN1bHRzOgogICAgcHJpbnQoJyUtMjJzICUtN3MgJS03cyAlLTlzICVzJyAlIChyLmdldCgndGFzaycpLCByLmdldCgnYWdlbnRfcGF0Y2hfcmMnKSwgci5nZXQoJ3Rlc3RfcGF0Y2hfcmMnKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgci5nZXQoJ3Jlc29sdmVkX2hlcmUnKSwgci5nZXQoJ2NvdW50cycpIG9yIChyLmdldCgnZXJyb3InKSBvciAnJylbOjQ1XSkpCm9rID0gc3VtKDEgZm9yIHIgaW4gcmVzdWx0cyBpZiByLmdldCgncmVzb2x2ZWRfaGVyZScpKQpwcmludCgnXG5yZXNvbHZlZCBoZXJlOiAlZCAvICVkJyAlIChvaywgbGVuKHJlc3VsdHMpKSkKcHJpbnQoJ3dyb3RlJywgT1VUKQo="""))

env = dict(os.environ, REPLAY_OUT='/kaggle/working/replay_result.json')
p = subprocess.run([sys.executable, str(RUNNER), str(TASKS), str(SNAPS), str(PD)] + sorted(PATCHES),
                   capture_output=True, text=True, env=env)
print(p.stdout)
print('STDERR', p.stderr[-1500:])
res = json.loads(Path('/kaggle/working/replay_result.json').read_text())
print('=== KAGGLE-SIDE REPLAY ===')
for r in res:
    print('%-22s resolved=%-6s counts=%s err=%s' % (r['task'], r.get('resolved_here'),
          r.get('counts'), (r.get('error') or '')[:40]))
print('resolved on Kaggle: %d / %d' % (sum(1 for r in res if r.get('resolved_here')), len(res)))
